# 04 Build / Validate V4 Model-Ready Targets

**Notebook version:** `SAJU_ML_BUILD_V4_MODEL_READY_TARGETS_20260816`

This notebook is the final target gate **before** a V4 feature/model tournament.

It does **not** load astrology feature caches, fit a model, tune hyperparameters, or open any sealed holdout.

It validates:

- exactly 100 verified same-axis opposite-polarity pair rows;
- source eligibility for every pair;
- pair-row and subject-macro chronology balance;
- mandatory age/calendar nuisance baselines;
- axis concentration;
- equal-subject pair weights;
- hash integrity of the frozen target package.

If any gate fails, output status is `V4_TARGET_HOLD` and model work must stop.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd

NOTEBOOK_VERSION = "SAJU_ML_BUILD_V4_MODEL_READY_TARGETS_20260816"

def find_repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/"saju_engine.py").exists(): return c
    raise FileNotFoundError("Run inside the saju repo")

REPO_ROOT=find_repo_root()
TARGET_DIR=REPO_ROOT/"research/ml_corpus/v4_target_expansion"
PAIRS=TARGET_DIR/"V4_VERIFIED_MATCHED_PAIRS_100.csv"
AUDIT=TARGET_DIR/"V4_TARGET_BALANCE_AUDIT.json"
MANIFEST=TARGET_DIR/"V4_TARGET_EXPANSION_FREEZE_MANIFEST.json"
OUT=REPO_ROOT/"research/ml/artifacts/v4_model_ready_targets"
OUT.mkdir(parents=True,exist_ok=True)

for p in [PAIRS,AUDIT,MANIFEST]:
    if not p.exists(): raise FileNotFoundError(p)

# This notebook intentionally has no NEW_CONFIRM / CHECK / FINAL / Validation B path.
print("NOTEBOOK VERSION:",NOTEBOOK_VERSION)
print("repo:",REPO_ROOT)

NOTEBOOK VERSION: SAJU_ML_BUILD_V4_MODEL_READY_TARGETS_20260816
repo: /Users/sangjinlee/Desktop/projects/saju


In [2]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

manifest=json.load(open(MANIFEST,encoding="utf-8"))
pairs=pd.read_csv(PAIRS)
expected=manifest["files"].get(PAIRS.name)
if expected and sha256(PAIRS)!=expected:
    raise RuntimeError("Pair CSV hash differs from frozen manifest")

assert manifest["new_confirm_loaded"] is False
assert manifest["sealed_sets_loaded"] is False
assert manifest["model_or_feature_scores_used_to_select_pairs"] is False
print("INTEGRITY PREFLIGHT: PASS")

INTEGRITY PREFLIGHT: PASS


In [3]:
required=["pair_id","subject_id","name","axis","positive_year","negative_year","source_pair_eligible","positive_earlier","age_only_pairwise","subject_equal_pair_weight"]
missing=[c for c in required if c not in pairs.columns]
if missing: raise RuntimeError("Missing columns: %s"%missing)

pairs["positive_year"]=pd.to_numeric(pairs["positive_year"],errors="raise").astype(int)
pairs["negative_year"]=pd.to_numeric(pairs["negative_year"],errors="raise").astype(int)
pairs["positive_earlier_calc"]=pairs["positive_year"]<pairs["negative_year"]
pairs["age_only_calc"]=np.where(pairs["positive_year"]<pairs["negative_year"],1.0,np.where(pairs["positive_year"]==pairs["negative_year"],0.5,0.0))

n=len(pairs)
unique_subjects=pairs["subject_id"].nunique()
pair_earlier=float(pairs["positive_earlier_calc"].mean())
subject_earlier=float(pairs.groupby("subject_id")["positive_earlier_calc"].mean().mean())
age_only=float(pairs["age_only_calc"].mean())
subject_age_only=float(pairs.groupby("subject_id")["age_only_calc"].mean().mean())
axis_shares=pairs["axis"].value_counts(normalize=True)
max_axis=float(axis_shares.max())

# Confirm subject-equal weights sum to 1 per person.
weight_sum=pairs.groupby("subject_id")["subject_equal_pair_weight"].sum()
weights_ok=bool(np.allclose(weight_sum.values,1.0))

gates={
 "n_pairs_at_least_100":n>=100,
 "unique_subjects_at_least_60":unique_subjects>=60,
 "pair_chronology_40_60":0.40<=pair_earlier<=0.60,
 "subject_macro_chronology_40_60":0.40<=subject_earlier<=0.60,
 "pair_age_only_40_60":0.40<=age_only<=0.60,
 "subject_macro_age_only_40_60":0.40<=subject_age_only<=0.60,
 "max_axis_share_le_45pct":max_axis<=0.45,
 "at_least_3_axes":pairs["axis"].nunique()>=3,
 "all_source_eligible":bool(pairs["source_pair_eligible"].astype(bool).all()),
 "no_same_year_pairs":bool((pairs["positive_year"]!=pairs["negative_year"]).all()),
 "subject_equal_weights":weights_ok,
}
status="V4_TARGET_MODEL_READY" if all(gates.values()) else "V4_TARGET_HOLD"

result={
 "version":"V4_MODEL_READY_TARGET_DECISION_V1",
 "notebook_version":NOTEBOOK_VERSION,
 "status":status,
 "metrics":{
   "n_pairs":n,"n_unique_subjects":int(unique_subjects),
   "pair_positive_earlier_share":pair_earlier,
   "subject_macro_positive_earlier_share":subject_earlier,
   "pair_age_only_pairwise":age_only,
   "subject_macro_age_only_pairwise":subject_age_only,
   "axis_counts":pairs["axis"].value_counts().to_dict(),
   "max_axis_share":max_axis,
 },
 "gates":gates,
 "next_rule":("Freeze target and proceed to V4 feature/model tournament with subject-level CV, subject-equal training weights, and Age-only/Calendar-year/Control baselines." if status=="V4_TARGET_MODEL_READY" else "STOP. Do not run model tournament."),
}
json.dump(result,open(OUT/"V4_MODEL_READY_TARGET_DECISION.json","w",encoding="utf-8"),ensure_ascii=False,indent=2)
pairs.to_csv(OUT/"V4_MODEL_READY_PAIRS.csv",index=False)
print(json.dumps(result,ensure_ascii=False,indent=2))

{
  "version": "V4_MODEL_READY_TARGET_DECISION_V1",
  "notebook_version": "SAJU_ML_BUILD_V4_MODEL_READY_TARGETS_20260816",
  "status": "V4_TARGET_MODEL_READY",
  "metrics": {
    "n_pairs": 100,
    "n_unique_subjects": 69,
    "pair_positive_earlier_share": 0.41,
    "subject_macro_positive_earlier_share": 0.45289855072463764,
    "pair_age_only_pairwise": 0.41,
    "subject_macro_age_only_pairwise": 0.45289855072463764,
    "axis_counts": {
      "COMPETITIVE": 43,
      "PROJECT": 40,
      "STATUS": 13,
      "RECOGNITION": 4
    },
    "max_axis_share": 0.43
  },
  "gates": {
    "n_pairs_at_least_100": true,
    "unique_subjects_at_least_60": true,
    "pair_chronology_40_60": true,
    "subject_macro_chronology_40_60": true,
    "pair_age_only_40_60": true,
    "subject_macro_age_only_40_60": true,
    "max_axis_share_le_45pct": true,
    "at_least_3_axes": true,
    "all_source_eligible": true,
    "no_same_year_pairs": true,
    "subject_equal_weights": true
  },
  "next_rule"

## If the result is `V4_TARGET_MODEL_READY`

The **next** notebook may perform the V4 feature/model tournament, but must obey:

1. subject-level outer CV;
2. training pair weights sum to one per subject;
3. `Age-only`, `Calendar-year-only`, and Production Control are mandatory benchmarks;
4. feature/model selection occurs inside outer training folds only;
5. no V2 `NEW_CONFIRM` or future holdout is opened during discovery.